# Final Task B 17-class - EfficientNet-B3

Train EfficientNet-B3 directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [2]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "task-B-17-new"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"task-B-17-new: {FINAL_B_DIR}")


ROOT: /workspace/woodbench
task-B-17-new: /workspace/woodbench/notebooks/phase 1/task-B-17-new


In [3]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    %pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [4]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "efficientnet_b3"
DISPLAY_NAME = "EfficientNet-B3"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


Backbone: EfficientNet-B3 (efficientnet_b3)
Batch size: 32
Data root: /workspace/woodbench/data/task-B-group-split


In [5]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Classes (17):
  00: Araucaria araucana
  01: Callitris columellaris
  02: Calocedrus rupestris
  03: Chamaecyparis obtusa
  04: Cryptomeria japonica
  05: Cunninghamia lanceolata
  06: Fokienia hodginsii
  07: Picea glauca
  08: Pinus cernua
  09: Pinus dalatensis
  10: Pinus krempfii
  11: Pinus radiata
  12: Podocarpus pilgeri
  13: Pseudotsuga brevifolia
  14: Taxus wallichiana
  15: Thujopsis dolabrata
  16: Xanthocyparis vietnamensis


In [6]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB
[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
[Device] cuda
[Output] /workspace/woodbench/data/experiments/Task-B-new/final

main/seed_42/efficientnet_b3
[main | efficientnet_b3 s42] train=3365, val=903, test=532, classes=17


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Epoch 1/50 | Train Loss: 2.8300, Acc: 7.32% | Val Loss: 2.8305, Acc: 5.76% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 5.76%)


Epoch 2/50 | Train Loss: 2.7565, Acc: 20.71% | Val Loss: 2.7080, Acc: 30.01% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 30.01%)


Epoch 3/50 | Train Loss: 2.5304, Acc: 49.91% | Val Loss: 2.4565, Acc: 59.36% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 59.36%)


Epoch 4/50 | Train Loss: 2.0710, Acc: 68.01% | Val Loss: 1.9991, Acc: 68.99% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to efficientnet_b3_seed42_best.pth (Val Acc: 68.99%)


... (281 outputs left out)

Epoch 47/50 | Train Loss: 1.0648, Acc: 99.08% | Val Loss: 1.1171, Acc: 95.90% | LR: 1.09e-06[EarlyStop 4/10]


Epoch 48/50 | Train Loss: 1.0595, Acc: 99.26% | Val Loss: 1.1166, Acc: 95.68% | LR: 4.87e-07[EarlyStop 5/10]


Epoch 49/50 | Train Loss: 1.0709, Acc: 98.63% | Val Loss: 1.1231, Acc: 95.35% | LR: 1.22e-07[EarlyStop 6/10]


Epoch 50/50 | Train Loss: 1.0690, Acc: 98.75% | Val Loss: 1.1179, Acc: 95.79% | LR: 0.00e+00[EarlyStop 7/10]
[taskB_label_smoothing_0.2 | efficientnet_b3 s42] val_acc=95.90% test_acc=97.56% F1=97.53%


In [7]:
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    96.12%    97.56%    97.53%    97.62%    97.57%
main                              123    96.46%    97.37%    97.43%    97.54%    97.46%
main                              456    96.01%    98.12%    98.16%    98.33%    98.14%
taskB_90rotation                   42    97.79%    98.68%    98.70%    98.83%    98.71%
taskB_label_smoothing_0.0          42    96.35%    96.99%    97.05%    97.17%    97.06%
taskB_label_smoothing_0.2          42    95.90%    97.56%    97.53%    97.62%    97.57%
----------------------------------------------------------------------------------------
main test_acc: 97.68 +/- 0.32%
main test_f1:  97.70 +/- 0.32%


## Continue after Vast interruption

Run the root, import/config, and data-check cells above, but do not rerun the old `run_all_experiments` cell. Then run each config cell below independently. Completed metrics are reused; interrupted checkpoints restore model, optimizer, and scheduler state.

In [8]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


In [9]:
from final_b_17class import EXPERIMENTS, load_completed_results, train_one

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


Loaded 6/6 completed configs for efficientnet_b3.


In [10]:
# Resume main seed 456 from its saved checkpoint
run_remaining("main", 456)


[main | efficientnet_b3 s456] Reusing /workspace/woodbench/data/experiments/Task-B-new/final/main/seed_456/efficientnet_b3/test_metrics.json


{'backbone': 'efficientnet_b3',
 'seed': 456,
 'experiment': 'main',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 96.01328903654485,
 'best_epoch': 43,
 'final_train_acc': 99.16666666666667,
 'final_val_acc': 95.45957918050941,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.981203007518797,
 'test_f1_macro': 0.9815636294004337,
 'test_precision_macro': 0.9832577488286832,
 'test_recall_macro': 0.9814150980296754,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 0.9850746268656716,
  'Calocedrus rupestris': 1.0,
  'Chamaecypa

In [11]:
# Remaining 90-degree rotation ablation
run_remaining("taskB_90rotation", 42)


[taskB_90rotation | efficientnet_b3 s42] Reusing /workspace/woodbench/data/experiments/Task-B-new/final/taskB_90rotation/efficientnet_b3/test_metrics.json


{'backbone': 'efficientnet_b3',
 'seed': 42,
 'experiment': 'taskB_90rotation',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 97.78516057585826,
 'best_epoch': 36,
 'final_train_acc': 99.07738095238095,
 'final_val_acc': 97.00996677740864,
 'stopped_early': True,
 'epochs_trained': 46,
 'test_accuracy': 0.9868421052631579,
 'test_f1_macro': 0.9870406564354994,
 'test_precision_macro': 0.9883075588957941,
 'test_recall_macro': 0.9871166415284063,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 0.9787234042553191,
  

In [12]:
# Remaining label-smoothing=0.0 ablation
run_remaining("taskB_label_smoothing_0.0", 42)


[taskB_label_smoothing_0.0 | efficientnet_b3 s42] Reusing /workspace/woodbench/data/experiments/Task-B-new/final/taskB_label_smoothing_0.0/efficientnet_b3/test_metrics.json


{'backbone': 'efficientnet_b3',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.0',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 96.34551495016612,
 'best_epoch': 36,
 'final_train_acc': 98.66071428571429,
 'final_val_acc': 95.79180509413068,
 'stopped_early': True,
 'epochs_trained': 46,
 'test_accuracy': 0.9699248120300752,
 'test_f1_macro': 0.9704612707241212,
 'test_precision_macro': 0.9717395908973259,
 'test_recall_macro': 0.9705862378279778,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 0.9538461538461539,
  'Calocedrus rupestris'

In [13]:
# Remaining label-smoothing=0.2 ablation
run_remaining("taskB_label_smoothing_0.2", 42)


[taskB_label_smoothing_0.2 | efficientnet_b3 s42] Reusing /workspace/woodbench/data/experiments/Task-B-new/final/taskB_label_smoothing_0.2/efficientnet_b3/test_metrics.json


{'backbone': 'efficientnet_b3',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.2',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 95.90254706533776,
 'best_epoch': 43,
 'final_train_acc': 98.75,
 'final_val_acc': 95.79180509413068,
 'stopped_early': False,
 'epochs_trained': 50,
 'test_accuracy': 0.9755639097744361,
 'test_f1_macro': 0.9752977480647076,
 'test_precision_macro': 0.976231094763831,
 'test_recall_macro': 0.9756804323685394,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 0.9850746268656716,
  'Calocedrus rupestris': 1.0,
  'Ch

In [14]:
results = list(continued_results.values())
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    96.12%    97.56%    97.53%    97.62%    97.57%
main                              123    96.46%    97.37%    97.43%    97.54%    97.46%
main                              456    96.01%    98.12%    98.16%    98.33%    98.14%
taskB_90rotation                   42    97.79%    98.68%    98.70%    98.83%    98.71%
taskB_label_smoothing_0.0          42    96.35%    96.99%    97.05%    97.17%    97.06%
taskB_label_smoothing_0.2          42    95.90%    97.56%    97.53%    97.62%    97.57%
----------------------------------------------------------------------------------------
main test_acc: 97.68 +/- 0.32%
main test_f1:  97.70 +/- 0.32%
